In [ ]:
# Prepare data for BERTopic models
# Create BALANCED training sets where each story has:
# - 1 human original (seed42)
# - 1 human large (sampled from 3 seeds: 42, 43, 44)
# - 1 model original (per model, seed42)
# - 1 model large (per model, seed42)
# 
# We create N bootstrap samples to account for human-large variance
# After training, we infer topics on ALL texts (including all 3 human-large seeds)

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import re
from collections import defaultdict

In [ ]:
# Path to JSONL files, output of linkappend model
data_dir = Path('../models/linkappend/data-out/conll_to_json')

jsonl_files = sorted(data_dir.glob('*_test_snapshots__local_json-nopound_pred.jsonlines'))
print(f"Found {len(jsonl_files)} JSONL files")
for f in jsonl_files:
    print(f"  {f.name}")

In [ ]:
def parse_filename(filename):
    """
    Parse filename to extract model, prompt type, and seed.
    Expected format: <model>_<prompt>_seed<N>_test_snapshots__local_json-nopound_pred.jsonlines
    """
    name = filename.replace('_test_snapshots__local_json-nopound_pred.jsonlines', '')
    pattern = r'^(.+?)_(large|original)_seed(\d+)$'
    
    match = re.match(pattern, name)
    if match:
        model = match.group(1)
        prompt_type = match.group(2)
        seed = match.group(3)
        return model, prompt_type, seed
    else:
        return name, 'unknown', None

def load_jsonl(filepath):
    """Load JSONL file and return list of records."""
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def clean_sentence(text):
    """
    Clean a sentence by removing pre-existing [SENT] markers.
    Handles both '[SENT]' and '[ SENT ]' variations.
    """
    # Remove [SENT] markers
    text = re.sub(r'\[\s*SENT\s*\]', '', text)
    # Clean up extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def format_story_for_bertopic(record):
    """Format a story record for BERTopic with [SENT] separators. The code is not ideal here, but it works."""
    sentences = record.get('sentences', [])
    descriptions = []
    for sent_tokens in sentences:
        # Join tokens and clean any pre-existing [SENT] markers
        sentence_text = ' '.join(sent_tokens)
        cleaned = clean_sentence(sentence_text)
        if cleaned:  # Only add non-empty sentences
            descriptions.append(cleaned)
    
    # Join descriptions with [SENT]
    story_text = ' [SENT] '.join(descriptions)
    
    # Ensure it ends with [SENT]
    if story_text and not story_text.endswith('[SENT]'):
        story_text += ' [SENT]'
    
    return story_text

def extract_story_id(doc_key):
    """Extract story_id from doc_key like 'doc_5201_59' -> 59."""
    if not doc_key:
        return None
    parts = doc_key.split('_')
    if len(parts) >= 3:
        try:
            return int(parts[-1])
        except ValueError:
            return None
    return None

# Test the parser
print("Testing filename parser:")
for f in jsonl_files[:3]:
    model, prompt, seed = parse_filename(f.name)
    print(f"  {f.name}")
    print(f"    -> Model: {model}, Prompt: {prompt}, Seed: {seed}")

# Test doc_key extraction
sample_records = load_jsonl(jsonl_files[0])
print(f"\nTesting doc_key extraction from {jsonl_files[0].name}:")
for i, rec in enumerate(sample_records[:3]):
    doc_key = rec.get('doc_key', '')
    story_id = extract_story_id(doc_key)
    print(f"  doc_key='{doc_key}' -> story_id={story_id}")

# Test the cleaning function
print()
print("Testing [SENT] cleaning:")
test_sentences = [
    "Tom looks around . [SENT]",
    "Bill walks away . [ SENT ]",
    "Normal sentence without markers .",
    "[SENT] Starting with marker ."
]
for sent in test_sentences:
    cleaned = clean_sentence(sent)
    print(f"  '{sent}' -> '{cleaned}'")

# Test full story formatting
print()
print("Sample formatted story:")
sample_story = format_story_for_bertopic(sample_records[0])
print(sample_story[:500] + "...")
print(f"\nNumber of [SENT] markers: {sample_story.count('[SENT]')}")
# Check for malformed markers
if '[ SENT ]' in sample_story:
    print("WARNING: Found malformed '[ SENT ]' markers!")
else:
    print("OK: No malformed markers found.")

In [ ]:
# Load ALL data and organize by (model, prompt, seed, story_id)
# Structure: data_dict[model][prompt][seed][story_id] = {'text': ..., 'metadata': ...}

data_dict = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

print("LOADING ALL FILES")

for jsonl_file in jsonl_files:
    model, prompt_type, seed = parse_filename(jsonl_file.name)
    records = load_jsonl(jsonl_file)
    
    print(f"Loading {jsonl_file.name}: {len(records)} stories (model={model}, prompt={prompt_type}, seed={seed})")
    
    for idx, record in enumerate(records):
        doc_key = record.get('doc_key', '')
        story_id = extract_story_id(doc_key)
        if story_id is None:
            story_id = idx  # Fallback to index if doc_key parsing fails
        
        story_text = format_story_for_bertopic(record)
        
        data_dict[model][prompt_type][seed][story_id] = {
            'text': story_text,
            'doc_key': doc_key,
            'story_id': story_id,
            'model_type': model,
            'prompt_type': prompt_type,
            'seed': seed,
            'num_sentences': len(record.get('sentences', [])),
            'source_file': jsonl_file.name
        }

# Print summary
print("DATA STRUCTURE SUMMARY")
print()
for model in sorted(data_dict.keys()):
    print(f"\n{model}:")
    for prompt in sorted(data_dict[model].keys()):
        seeds = sorted(data_dict[model][prompt].keys())
        n_stories = len(data_dict[model][prompt][seeds[0]])
        print(f"  {prompt}: seeds={seeds}, stories_per_seed={n_stories}")

In [ ]:
# Verify the data structure matches expectations:
# - human: original (seed42), large (seed42, seed43, seed44)
# - models: original (seed42), large (seed42 only)

models = sorted([m for m in data_dict.keys() if m != 'human'])
print(f"Models (non-human): {models}")
print(f"Human seeds for 'large': {sorted(data_dict['human']['large'].keys())}")
print(f"Human seeds for 'original': {sorted(data_dict['human']['original'].keys())}")

# Get story IDs from human original seed42
human_orig_seed = list(data_dict['human']['original'].keys())[0]  # Get first available seed
story_ids = sorted(data_dict['human']['original'][human_orig_seed].keys())
print(f"\nNumber of stories: {len(story_ids)}")
print(f"First 10 story IDs: {story_ids[:10]}")
print(f"Last 5 story IDs: {story_ids[-5:]}")

In [ ]:
# Process all JSONL files and create dataset
all_stories = []
all_metadata = []

for jsonl_file in jsonl_files:
    # Parse filename
    model, prompt_type, seed = parse_filename(jsonl_file.name)
    
    # Load data
    records = load_jsonl(jsonl_file)
    
    print(f"Processing {jsonl_file.name}: {len(records)} stories (model={model}, prompt={prompt_type}, seed={seed})")
    
    for idx, record in enumerate(records):
        # Format story text
        story_text = format_story_for_bertopic(record)
        
        # Extract metadata
        doc_key = record.get('doc_key', '')
        story_id = extract_story_id(doc_key)
        if story_id is None:
            story_id = idx  # Fallback to index
        num_descriptions = len(record.get('sentences', []))
        
        all_stories.append(story_text)
        all_metadata.append({
            'story_index': len(all_stories) - 1,
            'doc_key': doc_key,
            'story_id': story_id,
            'model_type': model,
            'prompt_type': prompt_type,
            'seed': seed,
            'num_story_images': num_descriptions,
            'source_file': jsonl_file.name
        })

print()
print(f"Total extracted stories: {len(all_stories)}")

# Create metadata DataFrame
metadata_df = pd.DataFrame(all_metadata)
print("\nMetadata summary:")
print(metadata_df.head(10))
print(f"\nModel types: {sorted(metadata_df['model_type'].unique())}")
print(f"Prompt types: {sorted(metadata_df['prompt_type'].unique())}")
print(f"Seeds: {sorted(metadata_df['seed'].dropna().unique())}")
print(f"\nStories per model:")
print(metadata_df['model_type'].value_counts().sort_index())

In [ ]:
# Save the formatted stories and metadata
canonical_output_dir = Path('/mimer/NOBACKUP/groups/naiss2024-6-297/cache/bertopic_data/bertopic_inputs')
canonical_output_dir.mkdir(parents=True, exist_ok=True)

output_stories_path = canonical_output_dir / 'all_stories_texts.json'
output_metadata_path = canonical_output_dir / 'all_stories_metadata.csv'

with open(output_stories_path, 'w') as f:
    json.dump(all_stories, f, indent=2)

# Save metadata as CSV
metadata_df.to_csv(output_metadata_path, index=False)

print(f"Saved {len(all_stories)} stories to {output_stories_path}")
print(f"Saved metadata to {output_metadata_path}")

print()
print("Verification")
print("Stories are formatted as list of strings")
print("Each story uses [SENT] separators between descriptions")
print(f"Metadata includes: {', '.join(metadata_df.columns)}")

print()
print("Sample story")
print(all_stories[0][:500] + "...")

In [ ]:
output_dir = Path('/mimer/NOBACKUP/groups/naiss2024-6-297/cache/bertopic_data/bertopic_inputs')
output_dir.mkdir(parents=True, exist_ok=True)

# Save all stories for inference
output_stories_path = output_dir / 'all_stories_texts.json'
output_metadata_path = output_dir / 'all_stories_metadata.csv'

with open(output_stories_path, 'w') as f:
    json.dump(all_stories, f, indent=2)

metadata_df.to_csv(output_metadata_path, index=False)

print(f"Saved {len(all_stories)} stories to {output_stories_path}")
print(f"Saved metadata to {output_metadata_path}")

In [ ]:
# For each bootstrap sample, randomly pick 1 of 3 human-large seeds per story

N_BOOTSTRAP = 10  # Number of balanced training sets to create
np.random.seed(42)

models = sorted([m for m in data_dict.keys() if m != 'human'])
human_large_seeds = sorted(data_dict['human']['large'].keys())  # ['42', '43', '44']
story_ids = sorted(data_dict['human']['original']['42'].keys())

print(f"Models: {models}")
print(f"Human-large seeds to sample from: {human_large_seeds}")
print(f"Number of stories: {len(story_ids)}")
print(f"Creating {N_BOOTSTRAP} balanced training sets...")

In [ ]:
def create_balanced_training_set(data_dict, story_ids, models, human_large_seed_choices):
    """
    Create a balanced training set.
    
    Args:
        data_dict: nested dict with all data
        story_ids: list of story IDs
        models: list of model names (excluding 'human')
        human_large_seed_choices: dict mapping story_id -> chosen seed for human-large
    
    Returns:
        texts: list of story texts
        metadata: list of metadata dicts
    """
    texts = []
    metadata = []
    
    for story_id in story_ids:
        human_orig = data_dict['human']['original']['42'][story_id]
        texts.append(human_orig['text'])
        metadata.append({
            **human_orig,
            'balanced_category': 'human_original'
        })
        
        chosen_seed = human_large_seed_choices[story_id]
        human_large = data_dict['human']['large'][chosen_seed][story_id]
        texts.append(human_large['text'])
        metadata.append({
            **human_large,
            'balanced_category': 'human_large',
            'sampled_seed': chosen_seed
        })
        
        for model in models:
            # Model original
            model_orig = data_dict[model]['original']['42'][story_id]
            texts.append(model_orig['text'])
            metadata.append({
                **model_orig,
                'balanced_category': 'model_original'
            })
            
            # Model large
            model_large = data_dict[model]['large']['42'][story_id]
            texts.append(model_large['text'])
            metadata.append({
                **model_large,
                'balanced_category': 'model_large'
            })
    
    return texts, metadata

In [ ]:
# Generate N bootstrap training sets
bootstrap_dir = output_dir / 'balanced_train_sets'
bootstrap_dir.mkdir(parents=True, exist_ok=True)

# Store which seed was chosen for each bootstrap sample (for reproducibility)
seed_choices_all = {}

for bootstrap_idx in range(N_BOOTSTRAP):
    # Randomly choose 1 of 3 seeds for human-large per story
    human_large_seed_choices = {
        story_id: np.random.choice(human_large_seeds)
        for story_id in story_ids
    }
    seed_choices_all[bootstrap_idx] = human_large_seed_choices
    
    # Create balanced set
    texts, metadata = create_balanced_training_set(
        data_dict, story_ids, models, human_large_seed_choices
    )
    
    # Save texts
    texts_path = bootstrap_dir / f'train_texts_bootstrap_{bootstrap_idx:02d}.json'
    with open(texts_path, 'w') as f:
        json.dump(texts, f, indent=2)
    
    # Save metadata
    meta_df = pd.DataFrame(metadata)
    meta_path = bootstrap_dir / f'train_metadata_bootstrap_{bootstrap_idx:02d}.csv'
    meta_df.to_csv(meta_path, index=False)
    
    if bootstrap_idx == 0:
        print(f"Bootstrap {bootstrap_idx}: {len(texts)} texts")
        print(f"  Categories: {meta_df['balanced_category'].value_counts().to_dict()}")
        print(f"  Models: {sorted(meta_df['model_type'].unique())}")

print()
print(f"Created {N_BOOTSTRAP} balanced training sets in {bootstrap_dir}")

In [ ]:
# Save seed choices for reproducibility
seed_choices_path = bootstrap_dir / 'human_large_seed_choices.json'
# Convert int keys to strings for JSON
seed_choices_json = {
    str(k): {str(sk): sv for sk, sv in v.items()}
    for k, v in seed_choices_all.items()
}
with open(seed_choices_path, 'w') as f:
    json.dump(seed_choices_json, f, indent=2)

print(f"Saved seed choices to {seed_choices_path}")